# Hướng dẫn Huấn luyện Mô hình Dịch Hán - Việt trên Kaggle GPU

Notebook này được thiết kế để bạn có thể import trực tiếp vào Kaggle hoặc Google Colab nhằm tinh chỉnh (fine-tune) mô hình dịch **`Helsinki-NLP/opus-mt-zh-vi`** sử dụng tập dữ liệu song song Hán - Việt thu được sau bước dóng hàng.

## Hướng dẫn chuẩn bị dữ liệu:
1. Chạy file `run_mapping.py` ở local để tạo các file tsv dóng hàng.
2. Chạy `python scripts/prepare_data.py` để tạo ra hai file dataset:
   * `output/translation_dataset/train.json`
   * `output/translation_dataset/val.json`
3. Tạo một **Dataset** mới trên Kaggle và tải hai file `train.json`, `val.json` lên đó.

## Bước 1: Kiểm tra cấu hình GPU và Cài đặt Thư viện

In [ ]:
# Kiểm tra thiết bị GPU có khả dụng không
!nvidia-smi

# Cài đặt các thư viện cần thiết cho Hugging Face
!pip install -q transformers[torch] datasets evaluate sacrebleu accelerate tensorboard

## Bước 2: Clone Repo Hugging Face và tải Script Huấn luyện chuẩn

Chúng ta sẽ không tự code hàm huấn luyện mà tải trực tiếp script `run_translation.py` chuẩn của Hugging Face Transformers.

In [ ]:
# Tải script run_translation.py chuẩn từ Hugging Face
!wget https://raw.githubusercontent.com/huggingface/transformers/main/examples/pytorch/translation/run_translation.py

# Kiểm tra xem tải thành công chưa
!ls -la run_translation.py

## Bước 3: Định nghĩa Đường dẫn Dữ liệu

Thay đổi đường dẫn `/kaggle/input/...` dưới đây cho khớp với tên Dataset mà bạn đã tạo trên Kaggle.

In [ ]:
import os

# ĐƯỜNG DẪN DỮ LIỆU TRÊN KAGGLE (Hãy cập nhật lại cho đúng tên Dataset của bạn)
DATASET_PATH = "/kaggle/input/sino-nom-translation-dataset/translation_dataset"

TRAIN_FILE = os.path.join(DATASET_PATH, "train.json")
VAL_FILE = os.path.join(DATASET_PATH, "val.json")

print(f"File Train tồn tại: {os.path.exists(TRAIN_FILE)}")
print(f"File Validation tồn tại: {os.path.exists(VAL_FILE)}")

## Bước 4: Chạy huấn luyện (Fine-tuning)

Ta chạy script huấn luyện sử dụng mô hình pre-trained `Helsinki-NLP/opus-mt-zh-vi`. Đây là mô hình dịch Trung-Việt rất nhẹ và tối ưu cho tác vụ dịch máy.

Các tham số cấu hình:
* `--model_name_or_path`: Mô hình pre-trained gốc (`Helsinki-NLP/opus-mt-zh-vi`).
* `--source_lang`: Ngôn ngữ nguồn (zh).
* `--target_lang`: Ngôn ngữ đích (vi).
* `--train_file` & `--validation_file`: Đường dẫn dữ liệu của bạn.
* `--output_dir`: Thư mục lưu checkpoint sau khi train.
* `--per_device_train_batch_size` & `--per_device_eval_batch_size`: Batch size (điều chỉnh theo bộ nhớ GPU).
* `--num_train_epochs`: Số epoch huấn luyện (ví dụ 5-10 epochs).
* `--save_steps`: Khoảng cách số step để lưu checkpoint.
* `--predict_with_generate`: Cho phép sinh văn bản tự động để tính metric BLEU.

In [ ]:
!python run_translation.py \
    --model_name_or_path Helsinki-NLP/opus-mt-zh-vi \
    --source_lang zh \
    --target_lang vi \
    --train_file {TRAIN_FILE} \
    --validation_file {VAL_FILE} \
    --output_dir /kaggle/working/han_viet_translation_model \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --do_train \
    --do_eval \
    --num_train_epochs 5 \
    --learning_rate 2e-5 \
    --weight_decay 0.01 \
    --predict_with_generate \
    --evaluation_strategy epoch \
    --save_strategy epoch

## Bước 5: Kiểm tra mô hình sau khi huấn luyện

Sau khi hoàn thành, ta có thể test trực tiếp khả năng dịch của checkpoint đã tinh chỉnh.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model_path = "/kaggle/working/han_viet_translation_model"
tokenizer = MarianTokenizer.from_pretrained(model_path)
model = MarianMTModel.from_pretrained(model_path)

def translate(text):
    # Mã hóa dữ liệu
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    # Dịch câu
    translated = model.generate(**inputs)
    # Giải mã
    return tokenizer.decode(translated[0], skip_special_tokens=True)

# Test một số câu Hán cổ mẫu
sample_han = "大南一統志卷之二承天府上"
print(f"Hán: {sample_han}")
print(f"Dịch: {translate(sample_han)}")